This document is inteded to showcase usage and allow the IBL team to acess example data until we decide about uploading the full dataset. The idea is to iterate and add improvemenets to the structure, suggest visualization and data access patterns that might be desirable for the final documentation.

To use it, you will need to run this notebook in an environment that has both pynwb and remfile installed plus any additional dependencies that we use for analysis such as matplotib, numpy, pandas, etc.

# Locate Examples

In [ ]:
import h5py
import remfile
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient

# Connect to DANDI and get the dandiset
dandiset_id = "000409"
client = DandiAPIClient()
dandiset = client.get_dandiset(dandiset_id, "draft")

# =============================================================================
# Session EIDs for NEW format files (desc-raw / desc-processed)
# =============================================================================

# Complete session with 2 probes - Session 1 (NYU-39, 2021-05-10, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_1 = "6ed57216-498d-48a6-b48b-a243a34710ea"

# Complete session with 2 probes - Session 2 (NYU-39, 2021-05-11, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_2 = "35ed605c-1a1a-47b1-86ff-2b56144f55af"

# Complete session with 1 probe (NYU-46, 2021-06-25, angelakilab)  
# - Full data: all videos, pose estimation, spike sorting for probe01
ONE_PROBE_SESSION_EID = "64e3fb86-928c-4079-865c-b364205b502e"

# Choose which session to use
session_eid = TWO_PROBE_SESSION_EID_1  # Change to TWO_PROBE_SESSION_EID_2 or ONE_PROBE_SESSION_EID

# =============================================================================
# Fetch assets by EID
# =============================================================================

# First, filter assets by EID
session_assets = [asset for asset in dandiset.get_assets() if session_eid in asset.path]

# Then, extract raw and processed files
raw_asset = next((asset for asset in session_assets if "desc-raw" in asset.path), None)
processed_asset = next((asset for asset in session_assets if "desc-processed" in asset.path), None)

print(f"Session EID: {session_eid}")
print(f"\nRaw file:       {raw_asset.path if raw_asset else 'Not found'}")
print(f"Processed file: {processed_asset.path if processed_asset else 'Not found'}")

# Raw NWBFile 

In [ ]:
s3_url = raw_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_raw = io.read()

nwbfile_raw

In [ ]:
probe_0_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe00AP"]
probe_1_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe01AP"]

print(f"Probe 0 AP data shape: {probe_0_ap_series.data.shape}")
print(f"Probe 1 AP data shape: {probe_1_ap_series.data.shape}")

probe_0_ap_series.electrodes.to_dataframe()

# Processed files

In [ ]:
# Stream the PROCESSED file (contains spike sorting, behavior, pose estimation)
print(f"Streaming PROCESSED file: {processed_asset.path}")

s3_url = processed_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_processed = io.read()

nwbfile_processed

In [ ]:
nwbfile_processed.trials.to_dataframe()

In [ ]:
nwbfile_processed.units.to_dataframe()